### Import libraries

In [12]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

### Set project root and load dataset

In [13]:
ROOT = Path.cwd()
if ROOT.name == "notebooks": ROOT = ROOT.parent
df = pd.read_csv(ROOT / "datasets/processed/model_ready_paddy.csv", keep_default_na=False, encoding="utf-8-sig")

### Get unique issue groups

In [14]:
issues = df[["issue_id", "label"]].drop_duplicates()

### Validate the dataset

In [15]:
assert issues["issue_id"].is_unique, "Each issue_id must map to one label."
assert issues["label"].value_counts().min() >= 3, "Some categories have too few issue groups for a stratified split."

### Split into Train, Validation and Test

In [16]:
trainval_issues, test_issues = train_test_split(
    issues,
    test_size=0.15,
    random_state=42,
    stratify=issues["label"]
)

### Split dataset

In [17]:
trainval_issues, test_issues = train_test_split(
    issues,
    test_size=0.15,
    random_state=42,
    stratify=issues["label"]
)

train_issues, val_issues = train_test_split(
    trainval_issues,
    test_size=16/90,
    random_state=42,
    stratify=trainval_issues["label"]
)

### Create split ID sets

In [18]:
split_ids = {
    "train": set(train_issues.issue_id),
    "validation": set(val_issues.issue_id),
    "test": set(test_issues.issue_id)
}
assert not (split_ids["train"] & split_ids["validation"])
assert not (split_ids["train"] & split_ids["test"])
assert not (split_ids["validation"] & split_ids["test"])

### Save the three datasets

In [19]:
for name, ids in split_ids.items():
    part = df[df["issue_id"].isin(ids)].copy()
    part.to_csv(ROOT / f"datasets/processed/paddy_{name}.csv", index=False, encoding="utf-8-sig")
    print(name, "rows:", len(part), "issue groups:", part.issue_id.nunique())

train rows: 17760 issue groups: 74
validation rows: 3840 issue groups: 16
test rows: 3840 issue groups: 16


In [20]:

assert not (split_ids["train"] & split_ids["validation"])
assert not (split_ids["train"] & split_ids["test"])
assert not (split_ids["validation"] & split_ids["test"])

for name, ids in split_ids.items():
    part = df[df["issue_id"].isin(ids)].copy()
    part.to_csv(ROOT / f"datasets/processed/paddy_{name}.csv", index=False, encoding="utf-8-sig")
    print(name, "rows:", len(part), "issue groups:", part.issue_id.nunique())

train = pd.read_csv(ROOT / "datasets/processed/paddy_train.csv", keep_default_na=False, encoding="utf-8-sig")
val = pd.read_csv(ROOT / "datasets/processed/paddy_validation.csv", keep_default_na=False, encoding="utf-8-sig")
test = pd.read_csv(ROOT / "datasets/processed/paddy_test.csv", keep_default_na=False, encoding="utf-8-sig")
print("\nTrain labels missing from validation:", sorted(set(train.label) - set(val.label)))
print("Train labels missing from test:", sorted(set(train.label) - set(test.label)))
print("Issue overlap train/test:", len(set(train.issue_id) & set(test.issue_id)))
assert len(set(train.issue_id) & set(val.issue_id)) == 0
assert len(set(train.issue_id) & set(test.issue_id)) == 0
assert len(set(val.issue_id) & set(test.issue_id)) == 0

train rows: 17760 issue groups: 74
validation rows: 3840 issue groups: 16
test rows: 3840 issue groups: 16

Train labels missing from validation: []
Train labels missing from test: []
Issue overlap train/test: 0


### Check missing labels

In [21]:
print("\nTrain labels missing from validation:", sorted(set(train.label) - set(val.label)))
print("Train labels missing from test:", sorted(set(train.label) - set(test.label)))


Train labels missing from validation: []
Train labels missing from test: []


### overlap checks

In [22]:
print("Issue overlap train/test:", len(set(train.issue_id) & set(test.issue_id)))
assert len(set(train.issue_id) & set(val.issue_id)) == 0
assert len(set(train.issue_id) & set(test.issue_id)) == 0
assert len(set(val.issue_id) & set(test.issue_id)) == 0

Issue overlap train/test: 0
